### ЗАДАЧА: Панель претензий к поставщикам по недопоставке по паттерну `MVC`

Команда procurement operations разбирает кейсы по недопоставке товара от поставщиков.
После приемки поставки система сравнивает ожидаемое количество и фактически полученный товар.
Если есть расхождение, создается кейс, по которому нужно провести расследование, запросить документы,
согласовать компенсацию от поставщика и закрыть кейс.

Нужно реализовать внутреннюю консольную панель по паттерну `MVC`, где:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за вывод данных;
- `Controller` принимает действия и связывает `Model` и `View`.

## Что должно храниться в кейсе

Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `supplier` — поставщик;
- `shipment_id` — идентификатор поставки;
- `sku` — товар;
- `expected_qty` — ожидаемое количество;
- `received_qty` — фактически принятое количество;
- `unit_cost` — себестоимость единицы товара;
- `claim_qty` — количество товара, которое считается недопоставленным;
- `claim_amount` — сумма претензии к поставщику;
- `approved_compensation` — согласованная компенсация;
- `remaining_loss` — остаток убытка после компенсации;
- `status` — текущий статус кейса;
- `manager` — сотрудник, который ведет кейс;
- `documents_verified` — подтверждены ли документы;
- `decision` — итоговое решение.

## Формулы

При создании кейса и после любого изменения компенсации нужно правильно считать:
- `claim_qty = expected_qty - received_qty`
- если `claim_qty < 0`, нужно выбрасывать ошибку;
- `claim_amount = claim_qty * unit_cost`
- `remaining_loss = claim_amount - approved_compensation`
- все денежные значения нужно округлять до 2 знаков.

## Статусы кейса

- `new`
- `investigating`
- `documents_requested`
- `documents_verified`
- `ready_for_resolution`
- `fully_compensated`
- `partial_compensated`
- `rejected`
- `escalated`

## Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `manager` несуществующему кейсу;
- финальные кейсы (`fully_compensated`, `partial_compensated`, `rejected`, `escalated`) нельзя менять дальше;
- начать расследование можно только из `new` и только если назначен `manager`;
- запросить документы можно только из `investigating`;
- подтвердить документы можно только из `documents_requested`;
- при подтверждении документов поле `documents_verified` должно стать `True`, а статус — `documents_verified`;
- установить `approved_compensation` можно только из `investigating` или `documents_verified`;
- `approved_compensation` не может быть меньше `0`;
- `approved_compensation` не может быть больше `claim_amount`;
- после изменения `approved_compensation` нужно пересчитать `remaining_loss`;
- перевод в `ready_for_resolution` возможен только из `investigating` или `documents_verified`;
- перевод в `ready_for_resolution` невозможен, если `approved_compensation == 0` и при этом `documents_verified == False`;
- завершить кейс как `fully_compensated` можно только из `ready_for_resolution`, если `approved_compensation == claim_amount`;
- завершить кейс как `partial_compensated` можно только из `ready_for_resolution`, если `0 < approved_compensation < claim_amount`;
- завершить кейс как `rejected` можно только из `ready_for_resolution`, если `approved_compensation == 0`;
- эскалировать кейс можно только из `investigating`, `documents_verified` или `ready_for_resolution`;
- при любом финальном статусе нужно записывать `decision`.

## Что должен уметь `Model`

Нужно самостоятельно спроектировать модель, но она должна уметь минимум:
- создавать кейс;
- назначать менеджера;
- начинать расследование;
- запрашивать документы;
- подтверждать документы;
- устанавливать `approved_compensation`;
- переводить кейс в `ready_for_resolution`;
- завершать кейс как `fully_compensated`;
- завершать кейс как `partial_compensated`;
- завершать кейс как `rejected`;
- эскалировать кейс;
- возвращать список кейсов;
- возвращать summary.

## Что должен уметь `View`

Нужно реализовать вывод:
- списка кейсов;
- summary;
- успешных сообщений;
- ошибок.

Если список кейсов пустой, вывести отдельное сообщение.

## Что должен делать `Controller`

Контроллер должен:
- вызывать методы модели;
- оборачивать операции в `try/except` с `ValueError`;
- передавать результат во view;
- обработать все действия из `actions`.

## Формат строки кейса

Каждый кейс можно вывести строкой такого вида:

`case_id | supplier | shipment_id | sku | expected_qty | received_qty | claim_qty | claim_amount | approved_compensation | remaining_loss | status | manager | documents_verified | decision`

## Что должно быть в summary

Нужно вернуть словарь, в котором есть:
- количество кейсов по статусам;
- `total_claim_amount` — общая сумма претензий;
- `total_approved_compensation` — общая согласованная компенсация;
- `total_remaining_loss` — общий остаток убытка;
- `verified_docs_cases` — количество кейсов, где документы подтверждены;
- `fully_compensated_amount` — сумма компенсаций по кейсам со статусом `fully_compensated`.

## Что нужно сделать в конце

1. Создать модель, view и controller.
2. Загрузить данные из `initial_cases`.
3. Обработать все действия из `actions`.
4. В конце вывести финальное состояние кейсов и summary.

In [2]:
from dataclasses import dataclass


initial_cases = [
    ("SC-100", "alpha-supply", "SHIP-7001", "SKU-100", 120, 102, 35.0),
    ("SC-101", "beta-distribution", "SHIP-7002", "SKU-200", 80, 80, 50.0),
]

actions = [
    ("show",),
    ("investigate", "SC-100"),
    ("assign", "SC-100", "Olga"),
    ("investigate", "SC-100"),
    ("docs_request", "SC-100"),
    ("docs_verify", "SC-100"),
    ("set_compensation", "SC-100", 500.0),
    ("ready", "SC-100"),
    ("partial", "SC-100", "supplier_accepted_partial_claim"),
    ("create", "SC-102", "gamma-warehouses", "SHIP-7003", "SKU-300", 60, 40, 28.0),
    ("assign", "SC-102", "Max"),
    ("investigate", "SC-102"),
    ("set_compensation", "SC-102", 560.0),
    ("ready", "SC-102"),
    ("full", "SC-102", "full_compensation_approved"),
    ("create", "SC-103", "delta-trade", "SHIP-7004", "SKU-400", 45, 30, 42.0),
    ("assign", "SC-103", "Ina"),
    ("investigate", "SC-103"),
    ("escalate", "SC-103", "supplier_disputes_shortage"),
    ("show",),
]

@dataclass
class ClaimCase:
    case_id: str
    supplier: str
    shipment_id: str
    sku: str
    expected_qty: int
    received_qty: int
    unit_cost: float
    claim_qty: int = 0
    claim_amount: float = 0.0
    approved_compensation: float = 0.0
    remaining_loss: float = 0.0
    status: str = "new"
    manager: str | None = None
    documents_verified: bool = False
    decision: str | None = None

    def __post_init__(self):
        self._calculate_claim()

    def _calculate_claim(self):
        if self.expected_qty < self.received_qty:
            raise ValueError(f"Invalid quantities: expected_qty ({self.expected_qty}) < received_qty ({self.received_qty})")
        self.claim_qty = self.expected_qty - self.received_qty
        self.claim_amount = round(self.claim_qty * self.unit_cost, 2)
        self._update_remaining_loss()

    def _update_remaining_loss(self):
        self.remaining_loss = round(self.claim_amount - self.approved_compensation, 2)

class ClaimModel:
    def __init__(self):
        self.cases = {}

    def create_case(self, case_id: str, supplier: str, shipment_id: str, sku: str, expected_qty: int, received_qty: int, unit_cost: float) -> None:
        if case_id in self.cases:
            raise ValueError("Case with this ID already exists")
        case = ClaimCase(case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost)
        self.cases[case_id] = case

    def assign_manager(self, case_id: str, manager: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status in ["fully_compensated", "partial_compensated", "rejected", "escalated"]:
            raise ValueError("Cannot modify final case")
        case.manager = manager


    def start_investigation(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status != "new":
            raise ValueError("Can only start investigation from 'new' status")
        if not case.manager:
            raise ValueError("Manager must be assigned before investigation")
        case.status = "investigating"

    def request_documents(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status != "investigating":
            raise ValueError("Can only request documents from 'investigating' status")
        case.status = "documents_requested"

    def verify_documents(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status != "documents_requested":
            raise ValueError("Can only verify documents from 'documents_requested' status")
        case.documents_verified = True
        case.status = "documents_verified"

    def set_compensation(self, case_id: str, compensation: float) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status not in ["investigating", "documents_verified"]:
            raise ValueError("Can only set compensation from 'investigating' or 'documents_verified' status")
        if compensation < 0:
            raise ValueError("Compensation cannot be negative")
        if compensation > case.claim_amount:
            raise ValueError("Compensation cannot exceed claim amount")
        case.approved_compensation = round(compensation, 2)
        case._update_remaining_loss()

    def mark_ready_for_resolution(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status not in ["investigating", "documents_verified"]:
            raise ValueError("Can only mark ready from 'investigating' or 'documents_verified' status")
        if case.approved_compensation == 0 and not case.documents_verified:
            raise ValueError("Cannot mark ready without compensation or verified documents")
        case.status = "ready_for_resolution"

    def close_fully_compensated(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status != "ready_for_resolution":
            raise ValueError("Can only close from 'ready_for_resolution' status")
        if case.approved_compensation != case.claim_amount:
            raise ValueError("Compensation must equal claim amount for full compensation")
        case.status = "fully_compensated"
        case.decision = decision

    def close_partial_compensated(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status != "ready_for_resolution":
            raise ValueError("Can only close from 'ready_for_resolution' status")
        if not (0 < case.approved_compensation < case.claim_amount):
            raise ValueError("Invalid partial compensation amount")
        case.status = "partial_compensated"
        case.decision = decision

    def escalate_case(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status not in ["investigating", "documents_verified", "ready_for_resolution"]:
            raise ValueError("Can only escalate from 'investigating', 'documents_verified' or 'ready_for_resolution' status")
        case.status = "escalated"
        case.decision = decision

    def list_cases(self) -> list[str]:
        if not self.cases:
            return ["No cases available"]
        rows = []
        for case in self.cases.values():
            rows.append(
                f"{case.case_id} | {case.supplier} | {case.shipment_id} | {case.sku} | "
                f"{case.expected_qty} | {case.received_qty} | {case.claim_qty} | "
                f"{case.claim_amount} | {case.approved_compensation} | {case.remaining_loss} | "
                f"{case.status} | {case.manager} | {case.documents_verified} | {case.decision}"
            )
        return rows

    def summary(self) -> dict[str, float | int]:
        result = {
            "total_claim_amount": 0.0,
            "total_approved_compensation": 0.0,
            "total_remaining_loss": 0.0,
            "verified_docs_cases": 0,
            "fully_compensated_amount": 0.0,
        }
        status_counts = {}
        for case in self.cases.values():
            status = case.status
            status_counts[status] = status_counts.get(status, 0) + 1
            result["total_claim_amount"] += case.claim_amount
            result["total_approved_compensation"] += case.approved_compensation
            result["total_remaining_loss"] += case.remaining_loss
            if case.documents_verified:
                result["verified_docs_cases"] += 1
            if case.status == "fully_compensated":
                result["fully_compensated_amount"] += case.approved_compensation
        result.update(status_counts)
        return result

class ClaimView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        print("Claim cases:")
        for row in rows:
            print(row)

    @staticmethod
    def render_summary(summary: dict[str, float | int]) -> None:
        print("Summary:", summary)

    @staticmethod
    def render_success(message: str) -> None:
        print("SUCCESS:", message)

    @staticmethod
    def render_error(message: str) -> None:
        print("ERROR:", message)

class ClaimController:
    def __init__(self, model: ClaimModel, view: ClaimView):
        self.model = model
        self.view = view

    def create_case(self, case_id: str, supplier: str, shipment_id: str, sku: str, expected_qty: int, received_qty: int, unit_cost: float) -> None:
        try:
            self.model.create_case(case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost)
            self.view.render_success(f"Case {case_id} created")
        except ValueError as error:
            self.view.render_error(str(error))

    def assign_manager(self, case_id: str, manager: str) -> None:
        try:
            self.model.assign_manager(case_id, manager)
            self.view.render_success(f"Manager assigned to {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def start_investigation(self, case_id: str) -> None:
        try:
            self.model.start_investigation(case_id)
            self.view.render_success(f"Investigation started for {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def request_documents(self, case_id: str) -> None:
        try:
            self.model.request_documents(case_id)
            self.view.render_success(f"Documents requested for {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def verify_documents(self, case_id: str) -> None:
        try:
            self.model.verify_documents(case_id)
            self.view.render_success(f"Documents verified for {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def set_compensation(self, case_id: str, compensation: float) -> None:
        try:
            self.model.set_compensation(case_id, compensation)
            self.view.render_success(f"Compensation set for {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def mark_ready_for_resolution(self, case_id: str) -> None:
        try:
            self.model.mark_ready_for_resolution(case_id)
            self.view.render_success(f"Case {case_id} marked ready for resolution")
        except ValueError as error:
            self.view.render_error(str(error))

    def close_fully_compensated(self, case_id: str, decision: str) -> None:
        try:
            self.model.close_fully_compensated(case_id, decision)
            self.view.render_success(f"Case {case_id} closed as fully compensated")
        except ValueError as error:
            self.view.render_error(str(error))

    def close_partial_compensated(self, case_id: str, decision: str) -> None:
        try:
            self.model.close_partial_compensated(case_id, decision)
            self.view.render_success(f"Case {case_id} closed as partially compensated")
        except ValueError as error:
            self.view.render_error(str(error))


    def close_rejected(self, case_id: str, decision: str) -> None:
        try:
            self.model.close_rejected(case_id, decision)
            self.view.render_success(f"Case {case_id} closed as rejected")
        except ValueError as error:
            self.view.render_error(str(error))

    def escalate_case(self, case_id: str, decision: str) -> None:
        try:
            self.model.escalate_case(case_id, decision)
            self.view.render_success(f"Case {case_id} escalated")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_cases(self) -> None:
        self.view.render_cases(self.model.list_cases())
        self.view.render_summary(self.model.summary())

model = ClaimModel()
view = ClaimView()
controller = ClaimController(model, view)

for case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost in initial_cases:
    model.create_case(case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost)

for action in actions:
    if action[0] == "show":
        controller.show_cases()
    elif action[0] == "create":
        _, case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost = action
        controller.create_case(case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost)
    elif action[0] == "assign":
        _, case_id, manager = action
        controller.assign_manager(case_id, manager)
    elif action[0] == "investigate":
        _, case_id = action
        controller.start_investigation(case_id)
    elif action[0] == "docs_request":
        _, case_id = action
        controller.request_documents(case_id)
    elif action[0] == "docs_verify":
        _, case_id = action
        controller.verify_documents(case_id)
    elif action[0] == "set_compensation":
        _, case_id, compensation = action
        controller.set_compensation(case_id, compensation)
    elif action[0] == "ready":
        _, case_id = action
        controller.mark_ready_for_resolution(case_id)
    elif action[0] == "partial":
        _, case_id, decision = action
        controller.close_partial_compensated(case_id, decision)
    elif action[0] == "full":
        _, case_id, decision = action
        controller.close_fully_compensated(case_id, decision)
    elif action[0] == "reject":
        _, case_id, decision = action
        controller.close_rejected(case_id, decision)
    elif action[0] == "escalate":
        _, case_id, decision = action
        controller.escalate_case(case_id, decision)

print("\n" + "="*80)
print("ФИНАЛЬНОЕ СОСТОЯНИЕ КЕЙСОВ И SUMMARY")
print("="*80)
controller.show_cases()

Claim cases:
SC-100 | alpha-supply | SHIP-7001 | SKU-100 | 120 | 102 | 18 | 630.0 | 0.0 | 630.0 | new | None | False | None
SC-101 | beta-distribution | SHIP-7002 | SKU-200 | 80 | 80 | 0 | 0.0 | 0.0 | 0.0 | new | None | False | None
Summary: {'total_claim_amount': 630.0, 'total_approved_compensation': 0.0, 'total_remaining_loss': 630.0, 'verified_docs_cases': 0, 'fully_compensated_amount': 0.0, 'new': 2}
ERROR: Manager must be assigned before investigation
SUCCESS: Manager assigned to SC-100
SUCCESS: Investigation started for SC-100
SUCCESS: Documents requested for SC-100
SUCCESS: Documents verified for SC-100
SUCCESS: Compensation set for SC-100
SUCCESS: Case SC-100 marked ready for resolution
SUCCESS: Case SC-100 closed as partially compensated
SUCCESS: Case SC-102 created
SUCCESS: Manager assigned to SC-102
SUCCESS: Investigation started for SC-102
SUCCESS: Compensation set for SC-102
SUCCESS: Case SC-102 marked ready for resolution
SUCCESS: Case SC-102 closed as fully compensated
S